In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [7]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [8]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [11]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [12]:
answer = assistant.rag(q['question'])

In [13]:
assistant.total_cost()

0.001227

In [14]:
print(answer)

Students don’t get the Zoom link directly.

To join an Office Hours or live workshop:
- Watch via **YouTube Live** on the **DataTalksClub YouTube channel**
- Look for the **video URL announcement** in **Telegram or Slack** before it starts
- Submit questions through **Slido**; the link is pinned in the chat when live

Don’t rely on posting questions in chat, since they may be missed.


In [15]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [16]:
assistant.total_cost()

0.001701

In [17]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [18]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [28]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [29]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. If you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [21]:
assistant.total_cost()

0.0021705

In [30]:
assistant.reset_usage()

In [23]:
assistant.total_cost()

0.0

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=4) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [26]:
results[:10]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still join if you missed the start date, but if you want a certificate, you need to finish with the live cohort and submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'answer_llm': 'Yes, as long 

In [27]:
df_results = pd.DataFrame(results)

In [28]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [29]:
assistant.total_cost()


0.34332825

In [ ]:
df_results.to_csv("data/rag-answers-new.csv", index=False)